# COMP8410 A1 - Week 4 (Nerfacto vs 3D Gaussian Splatting) - Colab
Runtime: **L4 or A100 GPU** (24 GB is enough). Upload the 105 bedroom photos once to Drive as `comp8410_a1/w4_data/setA/`.
Before saving: Edit > Notebook settings > tick *Omit code cell output when saving* (the outputs show a private room). Never use *Save a copy in GitHub*.
Order: Cell 1-2 setup, Cell 3 smoke tests, Cell 4 full run, Cell 5 results. **Disconnect the runtime when finished** (last cell).


## Cell 1: Drive, code, packages, COLMAP, nerfstudio


In [4]:
%cd /content
from google.colab import drive
import os
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/comp8410_a1'
for d in ['results', 'outputs', 'logs', 'w4_data']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

!rm -rf /content/a1 && git clone https://github.com/yash1998-MQ-AI/COMP8410-A1-48576158.git /content/a1
%cd "/content/a1/48576158 Evaluation"
!pip -q install -r requirements.txt
!pip -q install lpips tabulate imageio imageio-ffmpeg
!apt-get -qq install -y colmap ffmpeg > /dev/null 2>&1; colmap -h 2>&1 | head -1
!python pairs/w4_nerf_3dgs/install_nerfstudio.py 2>&1 | tail -6


Mounted at /content/drive
Cloning into '/content/a1'...
remote: Enumerating objects: 219, done.
remote: Counting objects: 100% (219/219), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 219 (delta 85), reused 216 (delta 82), pack-reused 0 (from 0)
Receiving objects: 100% (219/219), 428.45 KiB | 10.45 MiB/s, done.
Resolving deltas: 100% (85/85), done.
/content/a1/48576158 Evaluation
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.2 MB/s eta 0:00:00
/bin/bash: line 1: colmap: command not found
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
spopt 0.7.0 requires shapely>=2.1.0, but you have shapely 2.0.7 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
installing 44 requirements
wrote /usr/local/lib/python3.12/dist-packages/fpsample.py


In [5]:
!apt-get -qq update && apt-get -qq install -y colmap ffmpeg 2>&1 | tail -5
!which colmap && colmap -h 2>&1 | head -1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/usr/bin/colmap
COLMAP 3.7 -- Structure-from-Motion and Multi-View Stereo


## Cell 2: link Drive folders, GPU check, import check
The import check is the part most likely to fail on Colab's newest Python (nerfacc compiles CUDA code the first time it is imported, which can take a few minutes). If it fails, copy the last lines and stop here.


In [6]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!rm -rf results outputs
!ln -s {BASE}/results results
!ln -s {BASE}/outputs outputs
!mkdir -p {BASE}/outputs/w4_figures
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q tests/test_w4.py
!python -c "import torch, nerfacc, gsplat, nerfstudio; print('torch', torch.__version__, 'cuda', torch.version.cuda); print('IMPORT CHECK PASSED')" 2>&1 | tail -15
!ns-train --help 2>&1 | head -3
!echo photos on Drive: && ls {BASE}/w4_data/setA | wc -l


name, memory.total [MiB]
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB
...........                                                              [100%]
11 passed in 0.75s
torch 2.11.0+cu128 cuda 12.8
IMPORT CHECK PASSED
2026-10-30 12:01:00.943596: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-30 12:01:00.975274: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
photos on Drive:
105


## Cell 3: smoke tests (30 photos, 200 iterations, 1 seed). Results go to separate smoke files, not the real ones.
Run the four cells in order. The last one prints SMOKE PASSED or SMOKE FAILED and shows one held-out view from each model.


In [7]:
# 3a. COLMAP on 30 photos (CPU, a few minutes). Prints how many photos it could place.
!python pairs/w4_nerf_3dgs/run_w4.py process --images {BASE}/w4_data/setA --limit 30 2>&1 | tee {BASE}/logs/w4_smoke_process.log | tail -25


ns-process-data images --data /content/w4_work/input --output-dir /content/w4_work/scene --no-gpu --matching-method exhaustive --camera-type perspective --num-downscales 1
[12:01:49] 🎉 Done copying images with prefix 'frame_'.                                        process_data_utils.py:348
[12:01:50] 🎉 Done extracting COLMAP features.                                                       colmap_utils.py:137
[12:01:51] 🎉 Done matching COLMAP features.                                                         colmap_utils.py:151
──────────────────────────────────────────────  💀 💀 💀 ERROR 💀 💀 💀  ───────────────────────────────────────────────
Error running command: colmap mapper --database_path /content/w4_work/scene/colmap/database.db --image_path 
/content/w4_work/scene/images --output_path /content/w4_work/scene/colmap/sparse 
--Mapper.ba_global_function_tolerance=1e-6
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
# 3b. train both models for 200 iterations
!python pairs/w4_nerf_3dgs/run_w4.py train --models nerfacto splatfacto --seeds 0 --iters 200 --results results/w4_smoke.jsonl --ns-out /content/ns_smoke --ignore-gate 2>&1 | tee {BASE}/logs/w4_smoke_train.log | tail -30


In [ ]:
# 3c. render the held-out views and score them
!python pairs/w4_nerf_3dgs/score_w4.py --models nerfacto splatfacto --seeds 0 --results results/w4_smoke.jsonl --ns-out /content/ns_smoke --out-root /content/smoke_out 2>&1 | tee {BASE}/logs/w4_smoke_score.log | tail -30


In [ ]:
# 3d. smoke verdict and one picture
import glob, matplotlib.pyplot as plt
from PIL import Image
from a1eval import load
rows = load('results/w4_smoke.jsonl')
need = {(m, k) for m in ('nerfacto', 'splatfacto') for k in ('psnr', 'ssim', 'lpips', 'train_time_s', 'render_fps')}
got = {(r.model, r.metric) for r in rows}
print('SMOKE PASSED' if need <= got else f'SMOKE FAILED, missing: {sorted(need - got)}')
for m in ('nerfacto', 'splatfacto'):
    t = {r.metric: r.value for r in rows if r.model == m and r.metric in ('train_time_s', 'render_fps', 'peak_vram_gb')}
    print(m, {k: round(v, 2) for k, v in t.items()})
gt = sorted(glob.glob('/content/smoke_out/w4/gt/*.png'))
if gt:
    name = gt[0].rsplit('/', 1)[1]
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
    for a, (t, p) in zip(ax, [('real photo', gt[0]), ('Nerfacto (200 it)', f'/content/smoke_out/w4/nerfacto/s0/{name}'), ('3DGS (200 it)', f'/content/smoke_out/w4/splatfacto/s0/{name}')]):
        a.imshow(Image.open(p)); a.set_title(t); a.axis('off')
    plt.show()


## Cell 4: full run (all 105 photos, both models, 3 seeds, 30,000 iterations each)
COLMAP is run once. If its gate fails (fewer than 90% of the photos placed) the training cell refuses to start: fix the photos first rather than training on a broken scene. Training is resumable: finished runs are skipped.


In [ ]:
# 4a. COLMAP on all photos
!python pairs/w4_nerf_3dgs/run_w4.py process --images {BASE}/w4_data/setA 2>&1 | tee {BASE}/logs/w4_process.log | tail -25


In [ ]:
# 4b. train 2 models x 3 seeds
!python pairs/w4_nerf_3dgs/run_w4.py train --models nerfacto splatfacto --seeds 0 1 2 --ns-out /content/ns_runs 2>&1 | tee {BASE}/logs/w4_train.log | grep -v Warning | tail -30


In [ ]:
# 4c. render held-out views and score them
!python pairs/w4_nerf_3dgs/score_w4.py --models nerfacto splatfacto --seeds 0 1 2 --ns-out /content/ns_runs 2>&1 | tee {BASE}/logs/w4_score.log | tail -20


## Cell 5: tables, charts, pictures and the side-by-side clip
While connected, read this GPU's compute units per hour from the Resources panel and type it into `UNITS_PER_HR` so the training cost is priced. Keep *Omit code cell output when saving* ticked: the pictures show the room.


In [ ]:
UNITS_PER_HR = None   # e.g. 4.8, from the Resources panel
extra = f'--units-per-hr {UNITS_PER_HR}' if UNITS_PER_HR else ''
!python pairs/w4_nerf_3dgs/report_w4.py {extra} 2>&1 | tee {BASE}/logs/w4_report.log | grep -v Warning
!mkdir -p {BASE}/report_assets && cp -r report_assets/w4 {BASE}/report_assets/
!pip freeze > {BASE}/results/w4_ENV.txt

from IPython.display import Image as IPImage, Video, display
for f in ['w4_quality', 'w4_per_view_psnr', 'w4_cost']:
    display(IPImage(filename=f'report_assets/w4/{f}.png'))
display(IPImage(filename='outputs/w4_figures/w4_qualitative.jpg', width=1100))
import glob
clip = (glob.glob('outputs/w4_figures/w4_side_by_side.*') or [None])[0]
if clip:
    display(Video(clip, embed=True, width=900))


## Last cell: free the GPU
Set `AUTO_DISCONNECT = True` once everything above has finished and been copied to Drive. (Runtime > Manage sessions > Terminate does the same.)


In [2]:
AUTO_DISCONNECT = False
if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()
